# 08 — Timing and throughput measurement

This notebook confirms the logic used to measure forward-pass latency and throughput on a model. We apply warm-up iterations, take repeated timed measurements with a monotonic clock, and convert them to samples per second. To keep the demonstration trivial and deterministic on CPU we time small randomly initialised backbones at reduced width.

Modules exercised: `models/__init__.py` (`get_model`) and `pipelines/benchmark_pipeline/sizing.py` (`WidthScaler`) to build the trivial models; the timing methodology itself is the subject under test.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## Building trivial models to time

We instantiate a few backbones at small width and switch them to evaluation mode. These stand in for the size-matched models the real pipeline would time.

In [ ]:
from models import get_model
from pipelines.benchmark_pipeline.sizing import WidthScaler, _IMAGE_SIZE_MODELS

IN_CHANNELS  = 9
OUT_CHANNELS = 15
IMAGE_SIZE   = 32
BATCH        = 4

scaler = WidthScaler()
models = {}
for name in ["unet", "segformer", "dense_unet"]:
    config    = scaler.scaled_config(name, 0.15)
    overrides = {"in_channels": IN_CHANNELS, "out_channels": OUT_CHANNELS}
    if name in _IMAGE_SIZE_MODELS:
        overrides["image_size"] = IMAGE_SIZE
    model, _ = get_model(name, config=config, **overrides)
    models[name] = model.eval()

x = torch.randn(BATCH, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
print("models to time:", list(models))

## The measurement routine

The routine runs a few untimed warm-up passes (so allocation and caching costs are excluded), then times a number of repeats with `time.perf_counter`. It returns per-iteration latency statistics and throughput in samples per second.

In [ ]:
import time

def measure(model, x, warmup=3, repeats=20):
    with torch.no_grad():
        for _ in range(warmup):
            model(x)

        timings = []
        for _ in range(repeats):
            start = time.perf_counter()
            model(x)
            timings.append(time.perf_counter() - start)

    timings   = np.array(timings)
    per_iter  = float(timings.mean())
    return {
        "latency_ms_mean" : 1e3 * per_iter,
        "latency_ms_std"  : 1e3 * float(timings.std()),
        "throughput_sps"  : x.shape[0] / per_iter,
    }

results = {name: measure(model, x) for name, model in models.items()}
for name, stats in results.items():
    print(f"{name:12s} latency {stats['latency_ms_mean']:7.2f} +/- {stats['latency_ms_std']:5.2f} ms   "
          f"throughput {stats['throughput_sps']:8.1f} samples/s")

## Visual confirmation

The left panel shows mean per-iteration latency with its standard deviation as an error bar; the right panel shows the derived throughput. Latency and throughput are reciprocal up to the batch factor, so a model with higher latency should show lower throughput, which is the consistency check this plot makes visible.

In [ ]:
names      = list(results)
lat_mean   = [results[n]["latency_ms_mean"] for n in names]
lat_std    = [results[n]["latency_ms_std"] for n in names]
throughput = [results[n]["throughput_sps"] for n in names]

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 4.5))

ax_l.bar(names, lat_mean, yerr=lat_std, capsize=4, color="#a5533b")
ax_l.set_ylabel("latency per forward pass (ms)")
ax_l.set_title("Mean latency (warm-up excluded)")

ax_r.bar(names, throughput, color="#3b6ea5")
ax_r.set_ylabel("throughput (samples / s)")
ax_r.set_title("Derived throughput")

fig.tight_layout()
plt.show()

## Expected visual outcome

Each model reports a positive mean latency with a small standard deviation and a finite throughput. Across the two panels the ordering is consistent: the model with the largest latency bar has the smallest throughput bar, confirming the measurement logic is internally coherent.